In [1]:
import os
import subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import re
from tqdm import tqdm
import time

In [2]:
fastq_dir = Path("./fastq")
snp_only_dir = Path("./snp_only")
logs_dir = Path("./logs")
vcf_dir = Path("./vcf")
results_dir = Path("./results")
tmp_dir = Path("./tmp")

fastq_dir.mkdir(exist_ok=True)
snp_only_dir.mkdir(exist_ok=True)
logs_dir.mkdir(exist_ok=True)
vcf_dir.mkdir(exist_ok=True)
tmp_dir.mkdir(exist_ok=True)

In [3]:
# Group FASTQs by ENA run ID
samples = {}

for fq in fastq_dir.glob("*.fastq.gz"):
    name = fq.name

    match = re.search(r'([SE]RR\d+)', name)
    if not match:
        print(f"⚠️ Skipping {name}: no ENA run ID found.")
        continue

    clean_base = match.group(1)

    if clean_base not in samples:
        samples[clean_base] = {}

    if "_1" in name:
        suffix = "_1"
    elif "_2" in name:
        suffix = "_2"
    else:
        print(f"⚠️ Skipping {name}: no read pair info.")
        continue

    samples[clean_base][suffix] = fq.resolve()

print(f"✅ Found {len(samples)} samples to process.")

⚠️ Skipping SRR2024933.fastq.gz: no read pair info.
⚠️ Skipping SRR2024934.fastq.gz: no read pair info.
⚠️ Skipping SRR2024966.fastq.gz: no read pair info.
⚠️ Skipping SRR2024999.fastq.gz: no read pair info.
⚠️ Skipping SRR6650223.fastq.gz: no read pair info.
⚠️ Skipping SRR6650224.fastq.gz: no read pair info.
⚠️ Skipping SRR6650271.fastq.gz: no read pair info.
⚠️ Skipping SRR6650276.fastq.gz: no read pair info.
⚠️ Skipping SRR6650289.fastq.gz: no read pair info.
⚠️ Skipping SRR6650301.fastq.gz: no read pair info.
⚠️ Skipping SRR6650306.fastq.gz: no read pair info.
⚠️ Skipping SRR6650353.fastq.gz: no read pair info.
⚠️ Skipping SRR6650354.fastq.gz: no read pair info.
⚠️ Skipping SRR6650422.fastq.gz: no read pair info.
⚠️ Skipping SRR6650425.fastq.gz: no read pair info.
⚠️ Skipping SRR6797638.fastq.gz: no read pair info.
⚠️ Skipping SRR6797692.fastq.gz: no read pair info.
⚠️ Skipping SRR6797695.fastq.gz: no read pair info.
⚠️ Skipping SRR6831723.fastq.gz: no read pair info.
⚠️ Skipping 

In [4]:
def process_sample(sample, files):
    output_prefix = sample

    # Paths for checking
    result_txt = results_dir / f"{sample}.txt"
    result_json = results_dir / f"{sample}.json"
    snp_only_vcf = snp_only_dir / f"{sample}_snps_only.vcf"

    # Skip if output already exists
    if (result_txt.exists() or result_json.exists()) and snp_only_vcf.exists():
        print(f"⏩ Skipping {sample}: existing results and SNP VCF found.")
        return

    # Run TB-Profiler
    if "_1" in files and "_2" in files:
        fq1 = files["_1"]
        fq2 = files["_2"]
        tb_cmd = [
            "tb-profiler",
            "profile",
            "-1", str(fq1),
            "-2", str(fq2),
            "-p", str(output_prefix),
            "--txt",
            "--temp", "./tmp"
        ]
    elif "_1" in files:
        fq1 = files["_1"]
        tb_cmd = [
            "tb-profiler",
            "profile",
            "-1", str(fq1),
            "-p", str(output_prefix),
            "--txt",
            "--temp", "./tmp"
        ]
    else:
        print(f"⚠️ Skipping {sample}: no valid FASTQ files.")
        return

    print(f"🔬 Running TB-Profiler for {sample}")
    with open(logs_dir / f"{sample}_tbprofiler.log", "w") as log_file:
        subprocess.run(tb_cmd, stdout=log_file, stderr=log_file, check=True)

    # Run bcftools
    # input_vcf = vcf_dir / f"{output_prefix}.targets.vcf.gz"
    # output_vcf = snp_only_vcf

    # bcf_cmd = [
    # "bcftools", "view",
    # "-v", "snps",
    # "-i", "DP>=5 && QUAL>=20 && AF>=0.75",
    # input_vcf,
    # "-o", str(output_vcf),
    # "--output-type", "v"
    # ]   

    # # bcf_cmd = [
    # #     "bcftools", "view",
    # #     "-v", "snps",
    # #     input_vcf,
    # #     "-o", str(output_vcf),
    # #     "--output-type", "v"
    # # ]

    # print(f"🧬 Extracting SNPs for {sample}")
    # with open(logs_dir / f"{sample}_bcftools.log", "w") as log_file:
    #     subprocess.run(bcf_cmd, stdout=log_file, stderr=log_file, check=True)

    print(f"✅ Finished: {sample}")

In [ ]:
# Run with parallel jobs 
num_workers = 4
with ThreadPoolExecutor(max_workers=num_workers) as executor:
    futures = []
    for sample, files in samples.items():
        futures.append(executor.submit(process_sample, sample, files))

    for future in as_completed(futures):
        try:
            future.result()
        except subprocess.CalledProcessError as e:
            print(f"❌ Error running command: {e}") # meron pa ring error like ERR4810720, ERR2199934, SRR6152742 figure out why

print("🎉 All samples done.")

🔬 Running TB-Profiler for ERR046771
🔬 Running TB-Profiler for ERR038257
🔬 Running TB-Profiler for ERR046840
🔬 Running TB-Profiler for ERR046847
✅ Finished: ERR046771
🔬 Running TB-Profiler for ERR046930
❌ Error running command: Command '['tb-profiler', 'profile', '-1', '/mnt/c/Users/Lenovo/OneDrive/ths-st1-tb/fastq/ERR046930_1.fastq.gz', '-2', '/mnt/c/Users/Lenovo/OneDrive/ths-st1-tb/fastq/ERR046930_2.fastq.gz', '-p', 'ERR046930', '--txt', '--temp', './tmp']' returned non-zero exit status 1.🔬 Running TB-Profiler for ERR046937

✅ Finished: ERR038257
🔬 Running TB-Profiler for ERR046965
✅ Finished: ERR046847
🔬 Running TB-Profiler for ERR067635
✅ Finished: ERR046840
🔬 Running TB-Profiler for ERR067660
✅ Finished: ERR046937
🔬 Running TB-Profiler for ERR067667
✅ Finished: ERR067660
🔬 Running TB-Profiler for ERR067717
✅ Finished: ERR067635
🔬 Running TB-Profiler for ERR067742
✅ Finished: ERR067667
🔬 Running TB-Profiler for ERR067745
✅ Finished: ERR046965
🔬 Running TB-Profiler for ERR108500
✅ Fi